In [1]:
import os
import re
import numpy as np
import tifffile
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial


def find_folders(root):
    """Return all subfolders that actually contain files."""
    out = set()
    for r, d, f in os.walk(root):
        if f:
            out.add(os.path.abspath(r))
    return sorted(out)


def build_nested_structure(paths, base_folder):
    """
    Returns a dict keyed by the relative animal folder, which includes the middle dirs.

    Example key: "Control Mice/6 Months/148"
    Value: { "148 Slide 1 Section A2": ["148 Slide 1 Section A2_t0_z0", ...] }
    """
    base_folder = os.path.abspath(base_folder)
    grouped = defaultdict(lambda: defaultdict(list))

    for p in paths:
        rel = os.path.relpath(p, base_folder)
        parts = rel.split(os.sep)

        # locate the animal id folder (numeric)
        idx = None
        for i, part in enumerate(parts):
            if part.isdigit():
                idx = i
                break
        if idx is None or idx + 2 >= len(parts):
            continue

        prefix = os.path.join(*parts[:idx]) if idx > 0 else ""
        animal_id = parts[idx]
        animal_rel = os.path.join(prefix, animal_id) if prefix else animal_id

        section_folder = parts[idx + 1]
        zfolder = parts[idx + 2]
        grouped[animal_rel][section_folder].append(zfolder)

    # sort by z and t if present for correct stacking order
    def tz_key(name):
        m = re.search(r"_t(\d+)_z(\d+)$", name)
        if m:
            return (int(m.group(1)), int(m.group(2)))
        m = re.search(r"_z(\d+)$", name)
        return (0, int(m.group(1)) if m else -1)

    for animal_rel in grouped:
        for section in grouped[animal_rel]:
            grouped[animal_rel][section] = sorted(grouped[animal_rel][section], key=tz_key)

    return dict(grouped)


def ensure_dir(p):
    if not os.path.exists(p):
        os.makedirs(p)


def should_skip(out_path):
    """Return True if the expected output already exists."""
    return os.path.exists(out_path) and os.path.getsize(out_path) > 0


def stack_one_section(base_folder, out_base, animal_rel, section, zfolders, out_dtype=np.uint16):
    """
    Worker that stacks all z folders for a single section and writes the output.
    Returns a message string describing the result.
    """
    out_folder = os.path.join(out_base, animal_rel, section)
    ensure_dir(out_folder)
    out_path = os.path.join(out_folder, f"{section}_2D_stacked.tif")

    if should_skip(out_path):
        return f"Skip existing: {out_path}"

    volume = []
    missing = 0
    for zf in zfolders:
        abs_folder = os.path.join(base_folder, animal_rel, section, zf)
        tif_path = os.path.join(abs_folder, f"{zf}_final_mask.tif")
        if os.path.exists(tif_path):
            img = tifffile.imread(tif_path)
            volume.append(img)
        else:
            missing += 1

    if not volume:
        return f"No slices found for {animal_rel} | {section}  missing {missing}"

    vol = np.stack(volume, axis=0)
    tifffile.imwrite(out_path, vol.astype(out_dtype))
    return f"Saved {out_path} with shape {vol.shape}  missing {missing}"


def stack_and_save_multiproc(nested, base_folder, out_base, out_dtype=np.uint16, max_workers=16):
    """
    Prepare tasks for each section and run them in parallel.
    Skips sections that already have an output file.
    """
    tasks = []
    for animal_rel, sections in nested.items():
        for section, zfolders in sections.items():
            tasks.append((animal_rel, section, zfolders))

    if not tasks:
        print("No work to do.")
        return

    worker = partial(stack_one_section, base_folder, out_base, out_dtype=out_dtype)

    print(f"Submitting {len(tasks)} sections with {max_workers} workers")
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(worker, animal_rel, section, zfolders)
                   for animal_rel, section, zfolders in tasks]
        for fut in as_completed(futures):
            try:
                msg = fut.result()
                print(msg)
            except Exception as e:
                print(f"Error in worker: {e}")


# main
if __name__ == "__main__":
    input_base = "/media/core/core_operations/ImageAnalysis/Core/Haoran/core_projects/multi-scale-cellpose/output_Jimin_20X/segmentation_2D_planes"
    output_base = "/media/core/core_operations/ImageAnalysis/Core/Haoran/core_projects/multi-scale-cellpose/output_Jimin_20X/segmentation_2D_stacked"

    folders = find_folders(input_base)
    nested = build_nested_structure(folders, input_base)
    stack_and_save_multiproc(nested, input_base, output_base, out_dtype=np.uint16, max_workers=16)


Submitting 255 sections with 16 workers
Skip existing: /media/core/core_operations/ImageAnalysis/Core/Haoran/core_projects/multi-scale-cellpose/output_Jimin_20X/segmentation_2D_stacked/Control Mice/3 Months/241/241 Slide 2 Section A2/241 Slide 2 Section A2_2D_stacked.tif
Skip existing: /media/core/core_operations/ImageAnalysis/Core/Haoran/core_projects/multi-scale-cellpose/output_Jimin_20X/segmentation_2D_stacked/Control Mice/3 Months/241/241 Slide 2 Section C2/241 Slide 2 Section C2_2D_stacked.tif
Skip existing: /media/core/core_operations/ImageAnalysis/Core/Haoran/core_projects/multi-scale-cellpose/output_Jimin_20X/segmentation_2D_stacked/Control Mice/3 Months/244/244 Slide 2 Section A1/244 Slide 2 Section A1_2D_stacked.tif
Skip existing: /media/core/core_operations/ImageAnalysis/Core/Haoran/core_projects/multi-scale-cellpose/output_Jimin_20X/segmentation_2D_stacked/Control Mice/3 Months/241/241 Slide 2 Section B2/241 Slide 2 Section B2_2D_stacked.tif
Skip existing: /media/core/core_